# Exercises: Swapping embeddings (guided)

Machine Learning for Digital Scholarly Editions. Build your BERTopic pipeline.

This is a more guided version of `02_swapping-embeddings.ipynb`, same exercises, same goal, but broken into smaller steps with more explanation of the pandas/Python tools involved along the way. If you're comfortable with pandas already, use the regular version instead, it'll move faster.

The main notebook worked with the DH Conference Abstracts, all in English. Here we switch corpus: the **Hugo Schuchardt Archive letters**, historical correspondence to and from the linguist Hugo Schuchardt, written in several different languages within the same file. That mix is exactly the situation BERTopic's multilingual support is meant for, and exactly what these exercises test.

Run each code cell as you go and check its output before moving to the next one.

## Setup

If you're on **Google Colab**, run the cell below to mount your Drive and move into the materials folder.

If you're running **locally**, skip/comment out the cell below and just make sure `letters_full.csv` is in your working directory.

In [35]:
# Colab only
from google.colab import drive
drive.mount('/content/drive')
%cd /content/drive/MyDrive/dse-ml-2026/materials/2026-09-24_thursday/11_wolff_bertopic_pipeline/exercises/

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/drive/MyDrive/dse-ml-2026/materials/2026-09-24_thursday/11_wolff_bertopic_pipeline/exercises


### Exercise 1. Build a genuinely multilingual sample, and a multilingual vectorizer

Before any code: why not just grab 2000 random rows? Most of this corpus is written in German (you'll see the exact numbers in step 1.2), so a blind random sample would mostly test "does BERTopic handle German text", not "does it find themes that are actually shared across languages", which is the real question these exercises are about. The `keywords` column gives us a way to check for real cross-lingual overlap before sampling: it holds subject tags added by archivists, separated by semicolons, and the same subject sometimes gets tagged on letters written in several different languages. We'll use that to build a sample that's actually testing the right thing.

We're going to build this in small pieces:
1. Load the data and look at it.
2. Find which keywords actually appear across several languages.
3. Use that to filter the data down to documents that are part of a real cross-lingual subject.
4. Sample a balanced set of documents from that filtered data.
5. Build a stopword list covering all the languages involved.

#### 1.1 Load and look

`pd.read_csv(path)` reads a CSV file into a DataFrame, pandas' table-like structure. `.head()` shows you the first 5 rows, always a good habit right after loading anything, just to see what you're actually working with.

In [37]:
# load letters_full.csv into df, then look at df.head()
import pandas as pd
df = pd.read_csv("/content/drive/MyDrive/dse-ml-2026/materials/datasets/hsa/letters_full.csv")
df.head()

,source_file,pid,sender_id,sender,receiver_id,receiver,date,text,language,keywords,word_count
0,hsa.letter.1.xml,L.1-1,https://gams.uni-graz.at/o:hsa.persons#P.1069,Baissac,https://gams.uni-graz.at/o:hsa.persons#P.109,Schuchardt,1885-01-20,Ma Doudou vous envoie une petite brochure jaun...,fr,NaN,17
1,hsa.letter.1.xml,L.1-2,https://gams.uni-graz.at/o:hsa.persons#P.1069,Baissac,https://gams.uni-graz.at/o:hsa.persons#P.109,Schuchardt,1885-01-20,Nous sommes anxieux l’un et l’autre d’avoir de...,fr,NaN,31
2,hsa.letter.3.xml,L.3-1,https://gams.uni-graz.at/o:hsa.persons#P.1068,Bähr,https://gams.uni-graz.at/o:hsa.persons#P.109,Schuchardt,1924-04-22,Bereits 5 Tage nach Abgang meines letzten Brie...,de,Revue de Linguistique Romane; Verlage; Euskalt...,124
3,hsa.letter.3.xml,L.3-2-1,https://gams.uni-graz.at/o:hsa.persons#P.1068,Bähr,https://gams.uni-graz.at/o:hsa.persons#P.109,Schuchardt,1924-04-22,"Was die deutsche Verlagswelt angeht, so muss i...",de,Revue de Linguistique Romane; Verlage; Euskalt...,511
4,hsa.letter.3.xml,L.3-2-2,https://gams.uni-graz.at/o:hsa.persons#P.1068,Bähr,https://gams.uni-graz.at/o:hsa.persons#P.109,Schuchardt,1924-04-22,"sicherlich viel verdankt— erklärt, er wolle di...",de,Revue de Linguistique Romane; Verlage; Euskalt...,46


#### 1.2 How skewed is the language column?

`len(df)` gives you the number of rows. `df["language"].value_counts()` counts how many times each distinct value shows up in that column, sorted from most to least common. This is what shows us the problem we're about to work around: one language dominates the file.

In [38]:
# print len(df), then df.language.value_counts()
print(len(df))
print(df.language.value_counts())

42344
language
de     26790
fr      6362
it      3806
es      2858
pt      1003
en       944
idb      185
hu       142
eu       130
nl        24
roa       21
cy        20
la        16
ca        14
da        10
ro         9
ft         5
lad        2
pap        1
io         1
ms         1
Name: count, dtype: int64


#### 1.3 Drop rows with no keywords

Not every letter has a `keywords` tag. We can't use a row to check for cross-lingual overlap if it has no keyword to check, so we'll set those rows aside for this part. `df.dropna(subset=["keywords"])` returns a copy of `df` with only the rows where `keywords` is *not* missing. Store that copy under a new name, don't overwrite `df`, you'll still want the full thing later.

In [40]:
# df_with_keywords = df.dropna(subset=["keywords"]); print how many rows are left
df_with_keywords = df.dropna(subset=["keywords"])
print(len(df_with_keywords))

23314


#### 1.4 See what a keywords cell actually looks like

Before writing code to process every row, look at just one. `.iloc[0]` gets the first row by position, and `["keywords"]` gets that one column's value from it. Print it, and read it, it's several subject tags separated by `;`.

In [41]:
# print df_with_keywords.iloc[0]["keywords"]
df_with_keywords.iloc[0]["keywords"]

'Revue de Linguistique Romane; Verlage; Euskaltzaindia - Real Academia de la Lengua Vasca - Académie de la Langue Basque; Kongresse und Versammlungen; Sociedad de Estudios Vascos; Nationalismus; Baskischsprachige Literatur; Revue internationale des études basques; Zeitschrift für romanische Philologie; Literaturblatt für germanische und romanische Philologie; Globus. Illustrierte Zeitschrift für Länder- und Völkerkunde; Gedicht; Euskara (Organ für die Interessen der "Baskischen Gesellschaft")'

#### 1.5 Turn one keywords string into a list

`"a;b;c".split(";")` (plain Python, not even pandas yet) turns a string into a list wherever it sees a `;`. Try it on a made-up example first, then try it on the real value you printed in 1.4.

In [42]:
# "Etymologie;Universität Graz".split(";")
# then try it on df_with_keywords.iloc[0]["keywords"]
df_with_keywords.iloc[0]["keywords"].split(";")

['Revue de Linguistique Romane',
 ' Verlage',
 ' Euskaltzaindia - Real Academia de la Lengua Vasca - Académie de la Langue Basque',
 ' Kongresse und Versammlungen',
 ' Sociedad de Estudios Vascos',
 ' Nationalismus',
 ' Baskischsprachige Literatur',
 ' Revue internationale des études basques',
 ' Zeitschrift für romanische Philologie',
 ' Literaturblatt für germanische und romanische Philologie',
 ' Globus. Illustrierte Zeitschrift für Länder- und Völkerkunde',
 ' Gedicht',
 ' Euskara (Organ für die Interessen der "Baskischen Gesellschaft")']

#### 1.6 Do that for every row at once, with `.str.split`

`.split(";")` above worked on a single string. To do the same thing to an entire column at once, pandas gives you `.str.split(";")`, the `.str` part means "apply this string method to every value in the column". Create a new column, `keyword_list`, holding these lists.

You should now have a `keyword_list` column where each cell is a list rather than a single string. Check it with `.head()`.

In [44]:
# df_with_keywords["keyword_list"] = df_with_keywords["keywords"].str.split(";")
# df_with_keywords.head()
df_with_keywords["keyword_list"] = df_with_keywords["keywords"].str.split(";")
df_with_keywords.head()

/tmp/ipykernel_862/1134761708.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_with_keywords["keyword_list"] = df_with_keywords["keywords"].str.split(";")


,source_file,pid,sender_id,sender,receiver_id,receiver,date,text,language,keywords,word_count,keyword_list
2,hsa.letter.3.xml,L.3-1,https://gams.uni-graz.at/o:hsa.persons#P.1068,Bähr,https://gams.uni-graz.at/o:hsa.persons#P.109,Schuchardt,1924-04-22,Bereits 5 Tage nach Abgang meines letzten Brie...,de,Revue de Linguistique Romane; Verlage; Euskalt...,124,"[Revue de Linguistique Romane, Verlage, Eusk..."
3,hsa.letter.3.xml,L.3-2-1,https://gams.uni-graz.at/o:hsa.persons#P.1068,Bähr,https://gams.uni-graz.at/o:hsa.persons#P.109,Schuchardt,1924-04-22,"Was die deutsche Verlagswelt angeht, so muss i...",de,Revue de Linguistique Romane; Verlage; Euskalt...,511,"[Revue de Linguistique Romane, Verlage, Eusk..."
4,hsa.letter.3.xml,L.3-2-2,https://gams.uni-graz.at/o:hsa.persons#P.1068,Bähr,https://gams.uni-graz.at/o:hsa.persons#P.109,Schuchardt,1924-04-22,"sicherlich viel verdankt— erklärt, er wolle di...",de,Revue de Linguistique Romane; Verlage; Euskalt...,46,"[Revue de Linguistique Romane, Verlage, Eusk..."
5,hsa.letter.3.xml,L.3-3,https://gams.uni-graz.at/o:hsa.persons#P.1068,Bähr,https://gams.uni-graz.at/o:hsa.persons#P.109,Schuchardt,1924-04-22,Die Hauptschuld an diesen Zuständen tragen die...,de,Revue de Linguistique Romane; Verlage; Euskalt...,246,"[Revue de Linguistique Romane, Verlage, Eusk..."
6,hsa.letter.3.xml,L.3-4-1,https://gams.uni-graz.at/o:hsa.persons#P.1068,Bähr,https://gams.uni-graz.at/o:hsa.persons#P.109,Schuchardt,1924-04-22,Über Azkue habe ich ziemlich die gleiche Meinu...,de,Revue de Linguistique Romane; Verlage; Euskalt...,511,"[Revue de Linguistique Romane, Verlage, Eusk..."


#### 1.7 One row per keyword, with `.explode()`

Right now each row still holds a *list* of keywords, we want one row *per keyword* instead, so we can count languages per individual keyword later. `.explode("keyword_list")` does exactly this: it takes a column of lists and turns each list item into its own row, copying every other column's value alongside it.

Try it first on a tiny made-up table so you can see the effect clearly on just a couple of rows:
```python
toy = pd.DataFrame({"id": [1, 2], "tags": [["a", "b"], ["c"]]})
toy.explode("tags")
```
Then do the same to `df_with_keywords`, keeping only the columns you actually need for the next steps: `keyword_list` and `language`. (`df_with_keywords[["keyword_list", "language"]].explode("keyword_list")` selects just those two columns first, then explodes.)

In [46]:
# try the toy example first


# then build keyword_language = df_with_keywords[["keyword_list", "language"]].explode("keyword_list")
keyword_language = df_with_keywords[["keyword_list", "language"]].explode("keyword_list")
keyword_language.head()

,keyword_list,language
2,Revue de Linguistique Romane,de
2,Verlage,de
2,Euskaltzaindia - Real Academia de la Lengua V...,de
2,Kongresse und Versammlungen,de
2,Sociedad de Estudios Vascos,de


#### 1.8 Clean up whitespace

Splitting `"Etymologie; Universität Graz"` on `;` leaves a leading space on the second keyword: `" Universität Graz"`. That stray space would make pandas treat `"Etymologie"` and `" Etymologie"` as two different keywords later on. `.str.strip()` removes leading/trailing whitespace from every value in a column, rename the column to `keyword` while you're at it since it's no longer a list.

In [50]:
# keyword_language = keyword_language.rename(columns={"keyword_list": "keyword"})
# keyword_language["keyword"] = keyword_language["keyword"].str.strip()
keyword_language = keyword_language.rename(columns={"keyword_list":"keywords"})
keyword_language.head()
keyword_language["keywords"] = keyword_language["keywords"].str.strip()

#### 1.9 Count distinct languages per keyword

`.groupby("keyword")` splits the table into one group per distinct keyword value. Following it with `["language"].nunique()` then counts, *within each group*, how many distinct languages appear (`.nunique()` = number of unique values, different from `.count()`, which would just count rows, including repeats of the same language).

The result is a small table: one row per keyword, with the number of distinct languages it appears in.

In [51]:
# keyword_language_counts = keyword_language.groupby("keyword")["language"].nunique()
# keyword_language_counts.head()
keyword_language_counts = keyword_language.groupby("keywords")["language"].nunique()
keyword_language_counts.head()

,language
keywords,
28. Versammlung deutscher Schulmänner und Philologen (Philologentag) in Leipzig (Mai 1872),2
32. General-Versammlung der Deutschen Anthropologischen Gesellschaft in Metz 1901,1
35. Allgemeine Versammlung der Deutschen Anthropologischen Gesellschaft in Greifswald 1904,1
42. Versammlung deutscher Schulmänner und Philologen (Philologentag) in Wien (1893),1
44. Versammlung deutscher Philologen und Schulmänner (Philologentag) in Dresden (1897),1


#### 1.10 Keep only keywords with 3+ languages

`keyword_language_counts >= 3` gives you `True`/`False` for every keyword; using that inside `[...]` keeps only the rows where it's `True`. `.index` then gets you just the keyword names (not the counts), and wrapping that in `set(...)` gives you a Python set, convenient for the membership check in the next step.

Print `len(cross_lingual_keywords)`, you should get 246.

In [52]:
# cross_lingual_keywords = set(keyword_language_counts[keyword_language_counts >= 3].index)
# print(len(cross_lingual_keywords))
cross_lingual_keywords = set(keyword_language_counts[keyword_language_counts >= 3].index)

ollama to generate label description

#### 1.11 Write a function that checks one row

Now we need to go back to the *original* (non-exploded) `df_with_keywords`, and for each row, ask: does this row's `keywords` field contain *any* keyword from `cross_lingual_keywords`?

This is a good case for writing your own small function, since the check is a few steps: split the string on `;`, strip whitespace from each piece, turn it into a set, and check if that set overlaps at all with `cross_lingual_keywords` (`set_a & set_b` gives you the shared elements; wrapping the result in `bool(...)` turns "any shared elements at all" into a plain `True`/`False`).

Try the logic on a single made-up string first:
```python
example = "Etymologie;SomeKeywordThatIsNotCrossLingual"
tags = {k.strip() for k in example.split(";")}
print(tags)
print(bool(tags & cross_lingual_keywords))
```
(`{k.strip() for k in example.split(";")}` is a *set comprehension*, same idea as a for-loop that strips each piece and collects the results into a set, just written on one line.)

Once that makes sense, wrap the same logic in a function:
```python
def has_cross_lingual_keyword(keywords_field):
    # handle missing values first
    # split, strip, build a set of tags
    # return whether it overlaps with cross_lingual_keywords
```

In [58]:
# try the logic on the made-up example first


# then write has_cross_lingual_keyword(keywords_field) as a function
def has_cross_lingual_keyword(keywords_field):
  if pd.isna(keywords_field):
    return False
  tags = {k.strip() for k in keywords_field.split(";")}
  return (bool(tags & cross_lingual_keywords))

#### 1.12 Apply the function to every row

`.apply(some_function)` on a column runs that function once per value in the column and collects the results, one `True`/`False` per row here. Using that result inside `df_with_keywords[...]` keeps only the rows where it came back `True`.

You should end up with roughly 20,000 rows.

In [59]:
# df_cross_lingual = df_with_keywords[df_with_keywords["keywords"].apply(has_cross_lingual_keyword)]
# print(len(df_cross_lingual))
#df_with_keywords.head()
df_cross_lingual = df_with_keywords[df_with_keywords["keywords"].apply(has_cross_lingual_keyword)]
print(len(df_cross_lingual))

20338


#### 1.13 Sample the same number of rows from each language

We want exactly 333 rows from each of `de`, `fr`, `it`, `pt`, `es`, `en`, so that no single language dominates the final sample the way German dominates the raw file.

For one language at a time: `df_cross_lingual[df_cross_lingual.language == "de"]` keeps only the German rows, and `.sample(n=333, random_state=42)` picks 333 of them at random (`random_state` makes the "random" pick reproducible, same value every time the cell runs). Try it for just `"de"` first and check `len(...)` comes out to 333.

In [ ]:
# german_sample = df_cross_lingual[df_cross_lingual.language == "de"].sample(n=333, random_state=42)
# print(len(german_sample))
german_sample = df_cross_lingual[df_cross_lingual.language == 'de'].sample()

#### 1.14 Do that for all six languages, and combine them

Repeat 1.13 for each of `de`, `fr`, `it`, `pt`, `es`, `en`, collecting each language's 333-row sample into a Python list, then use `pd.concat(that_list)` to stick them all together into one DataFrame again.

A plain `for` loop works well here:
```python
balanced_languages = ["de", "fr", "it", "pt", "es", "en"]
samples = []
for lang in balanced_languages:
    # filter df_cross_lingual to this language, sample 333, append to samples

sample = pd.concat(samples)
```

In [ ]:
# build samples with a for loop, then sample = pd.concat(samples)


#### 1.15 Get the text out, and check the balance

`.text.to_list()` pulls the `text` column out of the DataFrame as a plain Python list, that's the `documents` list BERTopic actually wants. Then `sample.language.value_counts()` should show all six languages at exactly 333 each, confirming 1.14 worked.

In [ ]:
# documents = sample.text.to_list()
# print(len(documents), "documents")
# print(sample.language.value_counts())


#### 1.16 Download NLTK's stopword lists

We'll need stopwords (common words like "the", "und", "que" that don't carry topic meaning) for all six languages, not just one. NLTK ships stopword lists for many languages, but you have to download them once per machine first.

In [ ]:
# import nltk
# nltk.download('stopwords')


#### 1.17 Look at one language's list first

`from nltk.corpus import stopwords` gives you access to `stopwords.words("german")`, a plain Python list of German stopwords. Print its length and its first 10 entries just to see what you're working with, before combining six of these together.

In [ ]:
# from nltk.corpus import stopwords
# german_stopwords = stopwords.words("german")
# print(len(german_stopwords), german_stopwords[:10])


#### 1.18 Combine all six languages into one set

A `set` automatically drops duplicates, handy since some short words might coincidentally appear in more than one language's list. Start with an empty set, then loop over the six language names, adding each language's stopwords into it with `.update(...)` (`.update()` adds every item from a list into a set at once, unlike `.add()` which adds a single item).
```python
languages = ['german', 'french', 'italian', 'portuguese', 'spanish', 'english']
multilingual_stopwords = set()
for lang in languages:
    # add this language's stopwords into multilingual_stopwords
```

In [ ]:
# build multilingual_stopwords with the loop above, then print len(multilingual_stopwords)


#### 1.19 Build the vectorizer

You've already met `CountVectorizer` and `stop_words=` in the cleaning-stopwords notebook, the only difference here is passing a combined multilingual list instead of a single language. Build `vectorizer_model` now, you'll pass it into every `BERTopic(...)` from Exercise 2 onward, the same way `verbose=True` is just something you always include.

In [ ]:
# from sklearn.feature_extraction.text import CountVectorizer
# vectorizer_model = CountVectorizer(stop_words=list(multilingual_stopwords))


### Exercise 2. Run the default pipeline as a baseline

This part is exactly what you already did in notebook 1, `BERTopic(...)`, `.fit_transform(...)`, `.get_topic_info()`, just with `vectorizer_model` added in. Nothing new here conceptually, so this one moves faster.

#### 2.1 Create the model

Same `BERTopic(verbose=True, language="english")` pattern as notebook 1, add `vectorizer_model=vectorizer_model` as a third argument.

In [ ]:
# from bertopic import BERTopic
# topic_model_baseline = BERTopic(verbose=True, language="english", vectorizer_model=vectorizer_model)


#### 2.2 Fit it, and look at the topics

`.fit_transform(documents)` both trains the model and returns `topics` (one topic id per document) and `probs` (confidence per document). This step can take a minute or two, that's normal, watch the log lines it prints.

In [ ]:
# topics, probs = topic_model_baseline.fit_transform(documents)
# topic_model_baseline.get_topic_info()


### Exercise 3. Quantify how many documents ended up as outliers

We'll build the outlier-percentage formula piece by piece, so each part is visible before combining them.

#### 3.1 Get the topics table again

Same `get_topic_info()` as before, save it to a variable this time so you can work with it.

In [ ]:
# info = topic_model_baseline.get_topic_info()
# info


#### 3.2 Select just the outlier row

`info.Topic == -1` gives `True`/`False` for every row, `True` only where `Topic` is `-1`. `.loc[row_condition, column_name]` uses that to pick out just the `Count` value(s) where the condition holds, this is a more general version of the `df[df.language == "de"]` pattern from Exercise 1, `.loc` lets you filter rows *and* pick a column in one step.

In [ ]:
# info.loc[info.Topic == -1, "Count"]


#### 3.3 Turn that into a percentage

`.sum()` on the result of 3.2 gives you a single number (there's usually only one `-1` row anyway, but `.sum()` makes this work even if that ever changes). `info.Count.sum()` gives you the total across *all* topics, the denominator. Divide, multiply by 100, and `round(..., 1)` to one decimal place.

In [ ]:
# outlier_pct_baseline = round(info.loc[info.Topic == -1, "Count"].sum() / info.Count.sum() * 100, 1)
# print(f"{outlier_pct_baseline}% outliers (baseline)")


### Exercise 4. Try the multilingual shortcut

`BERTopic` accepts `language="multilingual"` instead of `"english"` for exactly this situation. Same pattern as Exercise 2, just that one argument changes, plus `vectorizer_model` as always.

In [ ]:
# topic_model_multilingual = BERTopic(verbose=True, language="multilingual", vectorizer_model=vectorizer_model)
# topics, probs = topic_model_multilingual.fit_transform(documents)
# topic_model_multilingual.get_topic_info()


### Exercise 5. Compare

Repeat the Exercise 3 outlier-percentage calculation for `topic_model_multilingual`, then look at a few topics side by side.

#### 5.1 Outlier percentage for the multilingual model

Same formula as 3.3, just on `topic_model_multilingual` this time.

In [ ]:
# info = topic_model_multilingual.get_topic_info()
# outlier_pct_multilingual = round(info.loc[info.Topic == -1, "Count"].sum() / info.Count.sum() * 100, 1)
# print(f"{outlier_pct_baseline}% outliers (baseline) vs {outlier_pct_multilingual}% (multilingual)")


#### 5.2 Compare one topic's words

`get_topic(0)` on each model gives you topic 0's word list (already familiar from notebook 1). Print both side by side and read them, do the words look like they belong together?

In [ ]:
# print("baseline:", topic_model_baseline.get_topic(0))
# print("multilingual:", topic_model_multilingual.get_topic(0))


#### 5.3 Do the same for a few more topics

Rather than repeat 5.2's two lines three more times by hand, a `for` loop over a small list of topic ids does it for you:
```python
for topic_id in [0, 1, 2]:
    print("baseline:", topic_model_baseline.get_topic(topic_id))
    print("multilingual:", topic_model_multilingual.get_topic(topic_id))
    print()
```
Note: topic ids are assigned independently by each model, "topic 0" in one has no guaranteed relationship to "topic 0" in the other, they're just each model's largest non-outlier topic. Judge by whether the words look coherent, not by matching ids.

In [ ]:
# loop over topic_id in [0, 1, 2] and print both models' get_topic(topic_id)


### Exercise 6. Reproduce the shortcut by hand

`language="multilingual"` is itself a shortcut: underneath, BERTopic loads a specific `SentenceTransformer` model. Let's load that same model ourselves and confirm it really is the same thing.

#### 6.1 Load the embedding model directly

`SentenceTransformer("model-name")` downloads (or reuses, if already downloaded) a specific pretrained model by name and loads it, ready to turn text into embeddings. This may take a moment the first time.

In [ ]:
# from sentence_transformers import SentenceTransformer
# embedding_model = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")


#### 6.2 Pass it into BERTopic manually

Instead of `language="multilingual"`, pass `embedding_model=embedding_model` directly. Everything else, `verbose`, `vectorizer_model`, stays the same as before.

In [ ]:
# topic_model_manual = BERTopic(verbose=True, embedding_model=embedding_model, vectorizer_model=vectorizer_model)
# topics, probs = topic_model_manual.fit_transform(documents)


#### 6.3 Compare topic counts

`get_topic_info()` returns one row per topic, so `len(...)` on it tells you how many topics (including the outlier row) each model found. These won't necessarily match exactly, BERTopic's default dimensionality-reduction step (UMAP) involves some randomness, so two separate fits on identical input can land on a similar but not identical number of topics.

In [ ]:
# print(len(topic_model_manual.get_topic_info()), "topics (manual)")
# print(len(topic_model_multilingual.get_topic_info()), "topics (Exercise 4, shortcut)")


### Exercise 7. Use a model the shortcut can't give you

`language="multilingual"` only ever gets you the one model from Exercise 4. `embedding_model=` is what actually lets you use a *different* one. Here we'll try `paraphrase-multilingual-mpnet-base-v2`, a larger multilingual model, the same kind of upgrade the main notebook does for English.

#### 7.1 Load the larger model and fit it

Same pattern as 6.1/6.2, just a different model name. This one is bigger, so expect it to take longer to embed the documents.

In [ ]:
# embedding_model_larger = SentenceTransformer("paraphrase-multilingual-mpnet-base-v2")
# topic_model_larger = BERTopic(verbose=True, embedding_model=embedding_model_larger, vectorizer_model=vectorizer_model)
# topics, probs = topic_model_larger.fit_transform(documents)


#### 7.2 Compare all three models at once

Checking `n_topics` and `outlier_pct` for three separate models means writing the same few lines three times, unless you loop. A list of `(name, model)` pairs lets you write the calculation once and run it for each pair. `info.Topic != -1` counts every *non*-outlier topic (`!=` means "not equal to"), pairing that with the outlier percentage matters: a model can post a great outlier percentage by collapsing everything into one or two giant topics rather than genuinely finding structure, so check both numbers together, not just one.
```python
for name, model in [
    ("baseline", topic_model_baseline),
    ("multilingual (MiniLM)", topic_model_multilingual),
    ("multilingual (mpnet)", topic_model_larger),
]:
    info = model.get_topic_info()
    n_topics = (info.Topic != -1).sum()
    outlier_pct = round(info.loc[info.Topic == -1, "Count"].sum() / info.Count.sum() * 100, 1)
    print(f"{name}: {n_topics} topics, {outlier_pct}% outliers")
```

In [ ]:
# run the loop above


#### 7.3 Read the actual words, not just the numbers

A low outlier percentage isn't automatically "better" if it comes from one huge, generic topic rather than several specific ones (check: does any one topic in `get_topic_info()` hold a very large share of `Count`?). And a higher outlier percentage isn't automatically "worse" either, if the topics it *does* find are genuinely coherent. Look at a few topics from `topic_model_larger` with `get_topic(topic_id)`: do the top words for any topic mix multiple languages around one recognizable subject (a name, a place, a field of study)? That's the thing only a multilingual embedding model can do, baseline structurally can't produce it. If you're looking for one specific theme, `topic_model_larger.find_topics("your search term")` searches semantically across the fitted topics and returns the closest matches by id, faster than scrolling the whole table.

In [ ]:
# inspect a few topics from topic_model_larger with get_topic(), and/or try find_topics()
